In [1]:
from AgentBasedModel.params_calibration.calibrationv2.utils_calibration_v2 import *
import pandas as pd
import random

In [6]:
file_10s = "ethusdt_10s_300days.csv"
prices_10s = get_prices(file_10s)

N = 60000
SLICE = 1000
num = ['mean_on_ret2', "std_on_ret2", "q1_on_ret2", "q5_on_ret2", "q95_on_ret2", "q99_on_ret2", "kurtosis_on_ret2", "skewness_on_ret"]
arrs = ["autocorrelation_on_ret", "autocorrelation_on_abs_ret"]

In [7]:
# посчитать среднее по метрикам из реального распределения + потом еще дисперсию, чтобы получился хорошая функция loss (нормированная адекватно)
df = pd.DataFrame()
step = 0
for i in range(N):
    step += 1
    if step % 1000 == 0:
        print(step)
    mx_start = len(prices_10s) - SLICE
    cur_start = random.randint(0, mx_start)
    cur_slice = prices_10s[cur_start:cur_start+SLICE]
    cur_params = pipeline(cur_slice, 0)
    ind = len(df)
    df.loc[ind, "start"] = int(cur_start)
    for key, val in cur_params.items():
        if key in arrs:
            idd = 0
            for v in val:
                df.loc[ind, f"{key}_lag_{idd}"] = v
                idd += 1
        else:
            df.loc[ind, key] = val

df

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000


,start,mean_on_ret2,std_on_ret2,q1_on_ret2,q5_on_ret2,q95_on_ret2,q99_on_ret2,kurtosis_on_ret2,skewness_on_ret,autocorrelation_on_ret_lag_0,...,autocorrelation_on_abs_ret_lag_11,autocorrelation_on_abs_ret_lag_12,autocorrelation_on_abs_ret_lag_13,autocorrelation_on_abs_ret_lag_14,autocorrelation_on_abs_ret_lag_15,autocorrelation_on_abs_ret_lag_16,autocorrelation_on_abs_ret_lag_17,autocorrelation_on_abs_ret_lag_18,autocorrelation_on_abs_ret_lag_19,autocorrelation_on_abs_ret_lag_20
0,1781964.0,0.001133,0.001517,-0.003693,-0.002715,0.001979,0.003320,0.875602,1.627885,1.0,...,0.025959,0.001290,0.030096,-0.054131,0.021878,-0.045569,0.030431,0.022385,0.058152,0.039118
1,395306.0,0.002347,0.003151,-0.003020,-0.002273,0.005408,0.011978,7.027711,0.938731,1.0,...,0.175934,0.190282,0.158275,0.149280,0.132678,0.120082,0.113994,0.089978,0.114807,0.103646
2,1867394.0,0.000487,0.000633,-0.001006,-0.000702,0.001070,0.001685,0.558934,0.665197,1.0,...,0.000962,0.081519,0.016549,0.011696,0.031794,0.023797,0.007045,-0.001427,0.017605,0.050892
3,543229.0,0.001607,0.002061,-0.003474,-0.002963,0.003802,0.004753,-0.323525,0.183515,1.0,...,0.237522,0.218088,0.161938,0.146297,0.181451,0.182714,0.142746,0.199623,0.119903,0.161668
4,1721719.0,0.001525,0.001976,-0.005750,-0.004038,0.002295,0.002614,0.769813,-2.422002,1.0,...,0.135815,0.141223,0.128776,0.201522,0.155766,0.154721,0.111033,0.162409,0.166016,0.107049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,920285.0,0.001212,0.001577,-0.003532,-0.002165,0.002282,0.003726,0.802267,-0.124984,1.0,...,0.111935,0.164300,0.138302,0.076953,0.057316,0.119056,0.073560,0.079598,0.124208,0.109520
59996,311745.0,0.000994,0.001327,-0.002514,-0.002290,0.002239,0.002796,-0.133730,-0.166722,1.0,...,0.043606,0.080797,0.068634,0.090974,0.103271,0.041082,-0.003402,0.093402,0.038036,0.092730
59997,2240763.0,0.001298,0.001986,-0.002430,-0.001946,0.003134,0.006902,6.667758,0.694316,1.0,...,0.152135,0.163961,0.113903,0.114940,0.155682,0.172636,0.156257,0.270125,0.109623,0.241281
59998,472989.0,0.001202,0.001583,-0.002670,-0.001943,0.002154,0.004253,1.471298,0.290760,1.0,...,0.083047,0.112794,0.093687,0.162147,0.160903,0.126572,0.172368,0.096435,0.081649,0.101402


In [8]:
# не хочется чтобы автокорреляция слишком много имела вес (потому что у нее столбец под каждый лаг
# и она сожрет весь вес, поэтому беру среднее по всем лагам
ret_ac_cols = [c for c in df.columns if c.startswith("autocorrelation_on_ret_lag_")]
abs_ac_cols = [c for c in df.columns if c.startswith("autocorrelation_on_abs_ret_lag_")]

base_cols = ['mean_on_ret2', "std_on_ret2", "q1_on_ret2", "q5_on_ret2", "q95_on_ret2", "q99_on_ret2", "kurtosis_on_ret2", "skewness_on_ret"]

df_summary = df[num].copy()
df_summary["autocorrelation_on_ret"] = df[ret_ac_cols].mean(axis=1)
df_summary["autocorrelation_on_abs_ret"] = df[abs_ac_cols].mean(axis=1)

df2 = (
    df_summary
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={"index": "param"})
)

df2

,param,mean,std
0,mean_on_ret2,0.001393,0.000763
1,std_on_ret2,0.001831,0.001072
2,q1_on_ret2,-0.003908,0.002759
3,q5_on_ret2,-0.002723,0.001709
4,q95_on_ret2,0.002743,0.001754
5,q99_on_ret2,0.003994,0.002886
6,kurtosis_on_ret2,1.028840,2.262870
7,skewness_on_ret,-0.012155,1.390985
8,autocorrelation_on_ret,0.328065,0.048495
9,autocorrelation_on_abs_ret,0.126264,0.054739


In [9]:
df2.to_csv("mean_std_ethusdt_validation_data2.csv", index=False)
df.to_csv("ethusdt_validation_data2.csv", index=False)